# LeetCode #178 — Rank Scores: xếp hạng "dày" bằng `DENSE_RANK()` hoặc truy vấn con đếm

**Mức độ:** Medium · **Chủ đề:** Database (SQL), Pandas

Skeleton trên LeetCode:

```sql
-- MySQL
# Write your MySQL query statement below
```

```python
# Pandas
import pandas as pd

def order_scores(scores: pd.DataFrame) -> pd.DataFrame:
```

Mục tiêu của notebook:

1. Hiểu yêu cầu xếp hạng: điểm cao hạng 1, **bằng điểm cùng hạng**, hạng sau **không bị nhảy**.
2. Hàm cửa sổ `DENSE_RANK() OVER (ORDER BY score DESC)`.
3. Cách không dùng hàm cửa sổ: hạng = số mức điểm **khác nhau** `>=` điểm của mình (truy vấn con tương quan).
4. Bẫy: `rank` là **từ khóa** trong MySQL 8 → phải đặt trong dấu nháy; `PARTITION BY` để xếp hạng trong từng nhóm.

# 1. Bài toán là gì?

Bảng `Scores`:

```text
+-------------+---------+
| Column Name | Type    |
+-------------+---------+
| id          | int     |
| score       | decimal |
+-------------+---------+
id là khóa chính. score có 2 chữ số sau dấu phẩy.
```

Xếp hạng các điểm:

- Sắp từ **cao xuống thấp**.
- Hai điểm **bằng nhau** cùng hạng.
- Sau một nhóm bằng nhau, hạng tiếp theo là **số nguyên liền kề** (không có "lỗ").

Trả về cột `score`, `rank`, sắp theo `score` giảm dần.

---

## Ví dụ

```text
Scores                      Kết quả
| id | score |              | score | rank |
| 1  | 3.50  |              | 4.00  | 1    |
| 2  | 3.65  |              | 4.00  | 1    |
| 3  | 4.00  |              | 3.85  | 2    |
| 4  | 3.85  |              | 3.65  | 3    |
| 5  | 4.00  |              | 3.65  | 3    |
| 6  | 3.65  |              | 3.50  | 4    |
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. SQL cơ bản — truy vấn bảng dữ liệu
2. Hàm cửa sổ (window function) trong SQL
3. Pandas DataFrame — SQL bằng Python

Cuối mục có phần **2.4 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 SQL cơ bản — truy vấn bảng dữ liệu

### Bảng, hàng, cột

Cơ sở dữ liệu quan hệ lưu dữ liệu trong các **bảng**. Mỗi **hàng** là một bản ghi, mỗi **cột** là một thuộc tính có kiểu (số, chuỗi…). Ô có thể **rỗng** — giá trị đặc biệt `NULL` ("không biết / không có").

```text
Employee
+----+-------+--------+--------------+
| id | name  | salary | departmentId |
+----+-------+--------+--------------+
| 1  | Joe   | 70000  | 1            |
| 2  | Jim   | 90000  | 1            |
| 3  | Henry | 80000  | 2            |
+----+-------+--------+--------------+
```

Trong notebook ta chạy SQL thật bằng **`sqlite3`** (có sẵn trong Python, chạy trong bộ nhớ). LeetCode dùng MySQL — cú pháp các câu cơ bản giống nhau, chỗ khác biệt sẽ được ghi chú.

In [1]:
import sqlite3

def sql_demo(query, params=()):
    con = sqlite3.connect(":memory:")
    con.execute("CREATE TABLE Employee (id INTEGER, name TEXT, salary INTEGER, departmentId INTEGER)")
    con.executemany("INSERT INTO Employee VALUES (?, ?, ?, ?)",
                    [(1, "Joe", 70000, 1), (2, "Jim", 90000, 1), (3, "Henry", 80000, 2),
                     (4, "Sam", 60000, 2), (5, "Max", 90000, 1)])
    con.execute("CREATE TABLE Department (id INTEGER, name TEXT)")
    con.executemany("INSERT INTO Department VALUES (?, ?)", [(1, "IT"), (2, "Sales")])
    cur = con.execute(query, params)
    cols = [d[0] for d in cur.description]
    out = cur.fetchall()
    con.close()
    print(cols)
    for r in out:
        print("  ", r)

sql_demo("SELECT * FROM Employee")

['id', 'name', 'salary', 'departmentId']
   (1, 'Joe', 70000, 1)
   (2, 'Jim', 90000, 1)
   (3, 'Henry', 80000, 2)
   (4, 'Sam', 60000, 2)
   (5, 'Max', 90000, 1)


### Thứ tự **viết** và thứ tự **thực hiện** của một câu SELECT

```text
Viết:        SELECT ... FROM ... WHERE ... GROUP BY ... HAVING ... ORDER BY ... LIMIT ...
Thực hiện:   FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT
```

Hiểu thứ tự thực hiện giải thích nhiều lỗi: `WHERE` chạy **trước** khi gom nhóm nên không dùng được hàm gộp (`COUNT`, `MAX`…) trong `WHERE` — phải dùng `HAVING`.

### Lọc, sắp xếp, giới hạn, bỏ trùng

In [2]:
print("Lương > 65000, giảm dần, lấy 2 hàng:")
sql_demo("SELECT name, salary FROM Employee WHERE salary > 65000 ORDER BY salary DESC LIMIT 2")
print("Các mức lương KHÁC NHAU, giảm dần, bỏ qua 1 hàng đầu (OFFSET):")
sql_demo("SELECT DISTINCT salary FROM Employee ORDER BY salary DESC LIMIT 1 OFFSET 1")

Lương > 65000, giảm dần, lấy 2 hàng:
['name', 'salary']
   ('Jim', 90000)
   ('Max', 90000)
Các mức lương KHÁC NHAU, giảm dần, bỏ qua 1 hàng đầu (OFFSET):
['salary']
   (80000,)


### Hàm gộp và GROUP BY

`COUNT`, `SUM`, `AVG`, `MIN`, `MAX` gộp nhiều hàng thành một giá trị. `GROUP BY cột` chia bảng thành nhóm theo cột đó, gộp riêng từng nhóm. `HAVING` lọc **sau** khi gộp.

In [3]:
sql_demo("""
SELECT departmentId, COUNT(*) AS n, MAX(salary) AS top
FROM Employee
GROUP BY departmentId
HAVING COUNT(*) >= 2
""")

['departmentId', 'n', 'top']
   (1, 3, 90000)
   (2, 2, 80000)


### NULL: "không có giá trị"

- Hàm gộp trên **tập rỗng** (trừ `COUNT`) trả về `NULL` — đây là mẹo để trả về `NULL` khi "không có đáp án".
- So sánh với `NULL` bằng `=` luôn **không đúng** → phải dùng `IS NULL`.
- `IFNULL(x, d)` / `COALESCE(x, d)` thay `NULL` bằng giá trị mặc định.

In [4]:
sql_demo("SELECT MAX(salary) AS m FROM Employee WHERE salary > 1000000")
sql_demo("SELECT NULL = NULL AS bang_nhau, NULL IS NULL AS la_null, COALESCE(NULL, 0) AS mac_dinh")

['m']
   (None,)
['bang_nhau', 'la_null', 'mac_dinh']
   (None, 1, 0)


### Truy vấn con (subquery)

Một câu `SELECT` có thể nằm **bên trong** câu khác: trả về một giá trị (dùng trong `WHERE x > (...)`), một danh sách (dùng với `IN`), hoặc một bảng tạm (đặt trong `FROM`).

In [5]:
print("Lương cao thứ hai = lương lớn nhất trong số những lương nhỏ hơn lương lớn nhất:")
sql_demo("SELECT MAX(salary) AS second FROM Employee WHERE salary < (SELECT MAX(salary) FROM Employee)")

Lương cao thứ hai = lương lớn nhất trong số những lương nhỏ hơn lương lớn nhất:
['second']
   (80000,)


### JOIN: ghép hai bảng

`A JOIN B ON điều_kiện` tạo các cặp hàng (a, b) thỏa điều kiện.

- **INNER JOIN** (mặc định): chỉ giữ cặp khớp.
- **LEFT JOIN**: giữ **mọi** hàng của A; hàng không khớp thì cột của B là `NULL`.
- **Self-join**: ghép bảng với **chính nó** (đặt hai bí danh khác nhau) — để so sánh các hàng trong cùng bảng.

In [6]:
sql_demo("""
SELECT d.name AS department, e.name AS employee, e.salary
FROM Employee e JOIN Department d ON e.departmentId = d.id
ORDER BY d.name, e.salary DESC
""")
print("Self-join: cặp nhân viên cùng phòng, người trước lương thấp hơn người sau")
sql_demo("""
SELECT a.name, b.name FROM Employee a JOIN Employee b
ON a.departmentId = b.departmentId AND a.salary < b.salary
""")

['department', 'employee', 'salary']
   ('IT', 'Jim', 90000)
   ('IT', 'Max', 90000)
   ('IT', 'Joe', 70000)
   ('Sales', 'Henry', 80000)
   ('Sales', 'Sam', 60000)
Self-join: cặp nhân viên cùng phòng, người trước lương thấp hơn người sau
['name', 'name']
   ('Joe', 'Jim')
   ('Joe', 'Max')
   ('Sam', 'Henry')


### Lỗi hay gặp

- Quên `DISTINCT` khi hỏi "giá trị lớn thứ k **khác nhau**".
- Dùng hàm gộp trong `WHERE` (phải dùng `HAVING` hoặc truy vấn con).
- `LIMIT 1 OFFSET k` trên bảng **không đủ** hàng trả về **không hàng nào** (không phải `NULL`) — bọc trong truy vấn con hoặc dùng `MAX` để có `NULL`.
- So sánh `= NULL` thay vì `IS NULL`.
- Trong MySQL, `LIMIT` không nhận biểu thức (`LIMIT N-1` lỗi) → phải tính trước (`SET N = N - 1`).

## 2.2 Hàm cửa sổ (window function) trong SQL

### Hàm gộp vs hàm cửa sổ

- **Hàm gộp + GROUP BY**: mỗi nhóm **co lại** thành một hàng.
- **Hàm cửa sổ** `f(...) OVER (PARTITION BY ... ORDER BY ...)`: **giữ nguyên** mọi hàng, mỗi hàng nhận thêm một giá trị tính trên "cửa sổ" các hàng liên quan.

```text
PARTITION BY departmentId   → chia thành các nhóm (giống GROUP BY nhưng không co lại)
ORDER BY salary DESC        → thứ tự trong mỗi nhóm
```

Có từ MySQL 8.0 và SQLite 3.25.

In [7]:
import sqlite3

def window_demo(query):
    con = sqlite3.connect(":memory:")
    con.execute("CREATE TABLE Scores (id INTEGER, score REAL, dept INTEGER)")
    con.executemany("INSERT INTO Scores VALUES (?, ?, ?)",
                    [(1, 3.50, 1), (2, 3.65, 1), (3, 4.00, 2), (4, 3.85, 2), (5, 4.00, 1), (6, 3.65, 2)])
    cur = con.execute(query)
    print([d[0] for d in cur.description])
    for r in cur.fetchall():
        print("  ", r)
    con.close()

### Ba hàm đánh số: ROW_NUMBER, RANK, DENSE_RANK

Khác nhau ở cách xử lý **giá trị bằng nhau**:

| giá trị | ROW_NUMBER | RANK | DENSE_RANK |
|---|---|---|---|
| 4.00 | 1 | 1 | 1 |
| 4.00 | 2 | 1 | 1 |
| 3.85 | 3 | **3** (nhảy) | **2** (liền) |

In [8]:
window_demo("""
SELECT score,
       ROW_NUMBER() OVER (ORDER BY score DESC) AS row_num,
       RANK()       OVER (ORDER BY score DESC) AS rnk,
       DENSE_RANK() OVER (ORDER BY score DESC) AS dense
FROM Scores ORDER BY score DESC
""")

['score', 'row_num', 'rnk', 'dense']
   (4.0, 1, 1, 1)
   (4.0, 2, 1, 1)
   (3.85, 3, 3, 2)
   (3.65, 4, 4, 3)
   (3.65, 5, 4, 3)
   (3.5, 6, 6, 4)


### PARTITION BY: đánh số riêng trong từng nhóm

"Top-k mỗi nhóm" = đánh số trong nhóm rồi lọc `<= k`. Vì `WHERE` chạy **trước** hàm cửa sổ, phải bọc trong truy vấn con hoặc `WITH`.

In [9]:
window_demo("""
WITH ranked AS (
    SELECT dept, id, score,
           DENSE_RANK() OVER (PARTITION BY dept ORDER BY score DESC) AS r
    FROM Scores
)
SELECT dept, id, score FROM ranked WHERE r = 1 ORDER BY dept, id
""")

['dept', 'id', 'score']
   (1, 5, 4.0)
   (2, 3, 4.0)


### LAG / LEAD: nhìn hàng trước / sau

`LAG(col, k)` lấy giá trị của hàng cách **trước** `k` hàng (theo `ORDER BY` của cửa sổ), `LEAD` lấy hàng **sau**; không có thì `NULL`.

In [10]:
window_demo("""
SELECT id, score,
       LAG(score, 1)  OVER (ORDER BY id) AS truoc,
       LEAD(score, 1) OVER (ORDER BY id) AS sau
FROM Scores ORDER BY id
""")

['id', 'score', 'truoc', 'sau']
   (1, 3.5, None, 3.65)
   (2, 3.65, 3.5, 4.0)
   (3, 4.0, 3.65, 3.85)
   (4, 3.85, 4.0, 4.0)
   (5, 4.0, 3.85, 3.65)
   (6, 3.65, 4.0, None)


### WITH (CTE): đặt tên cho bảng tạm

`WITH ten AS (SELECT ...) SELECT ... FROM ten` giúp chia truy vấn dài thành các bước dễ đọc — tương tự gán biến trung gian trong Python.

### Lỗi hay gặp

- Dùng `RANK` khi đề muốn thứ hạng **liền nhau** (cần `DENSE_RANK`), hoặc ngược lại.
- Lọc kết quả hàm cửa sổ trong `WHERE` cùng câu → lỗi; phải bọc truy vấn con / CTE.
- `rank` là **từ khóa** trong MySQL 8 → đặt tên cột là `` `rank` `` (backtick).
- Quên `PARTITION BY` → đánh số trên **toàn bảng** thay vì từng nhóm.

## 2.3 Pandas DataFrame — SQL bằng Python

### DataFrame là gì?

`pandas.DataFrame` là một **bảng** trong bộ nhớ: các cột có tên, mỗi cột là một `Series` (mảng có nhãn). LeetCode có phiên bản Pandas cho các bài SQL; mỗi thao tác SQL có một tương đương:

| SQL | Pandas |
|---|---|
| `WHERE cond` | `df[df["col"] > x]` |
| `SELECT a, b` | `df[["a", "b"]]` |
| `ORDER BY a DESC` | `df.sort_values("a", ascending=False)` |
| `DISTINCT` | `df.drop_duplicates()` / `series.unique()` |
| `GROUP BY` + gộp | `df.groupby("k")["v"].max()` |
| `JOIN` | `df1.merge(df2, left_on=..., right_on=...)` |
| `DENSE_RANK()` | `series.rank(method="dense", ascending=False)` |
| `LAG(col)` | `series.shift(1)` |

In [11]:
import pandas as pd

emp = pd.DataFrame({"id": [1, 2, 3, 4, 5],
                    "name": ["Joe", "Jim", "Henry", "Sam", "Max"],
                    "salary": [70000, 90000, 80000, 60000, 90000],
                    "departmentId": [1, 1, 2, 2, 1]})
dep = pd.DataFrame({"id": [1, 2], "name": ["IT", "Sales"]})
print(emp)

   id   name  salary  departmentId
0   1    Joe   70000             1
1   2    Jim   90000             1
2   3  Henry   80000             2
3   4    Sam   60000             2
4   5    Max   90000             1


### Lọc, chọn cột, sắp xếp, bỏ trùng

In [12]:
print(emp[emp["salary"] > 65000][["name", "salary"]].sort_values("salary", ascending=False).head(2))
print("lương khác nhau giảm dần:", sorted(emp["salary"].unique().tolist(), reverse=True))

  name  salary
1  Jim   90000
4  Max   90000
lương khác nhau giảm dần: [90000, 80000, 70000, 60000]


### groupby và transform

- `groupby(k)[v].max()` → **co** mỗi nhóm thành một giá trị (như `GROUP BY`).
- `groupby(k)[v].transform("max")` → trả về giá trị của nhóm **cho từng hàng** (giống hàm cửa sổ `MAX() OVER (PARTITION BY k)`), rất tiện để lọc "hàng có giá trị lớn nhất nhóm".

In [13]:
print(emp.groupby("departmentId")["salary"].max())
top = emp[emp["salary"] == emp.groupby("departmentId")["salary"].transform("max")]
print(top)

departmentId
1    90000
2    80000
Name: salary, dtype: int64
   id   name  salary  departmentId
1   2    Jim   90000             1
2   3  Henry   80000             2
4   5    Max   90000             1


### merge (JOIN), rank, shift

In [14]:
m = emp.merge(dep, left_on="departmentId", right_on="id", suffixes=("", "_dep"))
print(m[["name_dep", "name", "salary"]])
s = pd.Series([4.00, 4.00, 3.85, 3.65])
print("rank dense:", s.rank(method="dense", ascending=False).tolist(), "| rank min (như RANK):", s.rank(method="min", ascending=False).tolist())
print("shift(1) (như LAG):", s.shift(1).tolist())

  name_dep   name  salary
0       IT    Joe   70000
1       IT    Jim   90000
2    Sales  Henry   80000
3    Sales    Sam   60000
4       IT    Max   90000
rank dense: [1.0, 1.0, 2.0, 3.0] | rank min (như RANK): [1.0, 1.0, 3.0, 4.0]
shift(1) (như LAG): [nan, 4.0, 4.0, 3.85]


### Lỗi hay gặp

- `df[...]` lọc trả về DataFrame **có thể rỗng** — kiểm tra `len(...)` trước khi lấy `.iloc[0]`.
- `rank` trả về số thực (`1.0`) → `astype(int)` nếu cần.
- `merge` hai bảng cùng tên cột → cột bị thêm hậu tố `_x`, `_y` (hoặc `suffixes` tự đặt).
- Khi đề yêu cầu trả `null`, trả `None` trong một DataFrame một hàng — không trả DataFrame rỗng.

## 2.4 Áp dụng vào bài này

Xếp hạng "không có lỗ" (hai điểm bằng nhau cùng hạng, hạng tiếp theo là số liền sau) chính là **`DENSE_RANK() OVER (ORDER BY score DESC)`**. Không dùng hàm cửa sổ thì hạng của một điểm = **số điểm khác nhau `>=` nó** (truy vấn con `COUNT(DISTINCT ...)`). Nhớ `rank` là từ khóa trong MySQL. Bản Pandas: `rank(method="dense", ascending=False)`.

# 3. Nền tảng

In [15]:
import sqlite3
import random
import pandas as pd


def run_sql(sql: str, tables: dict, params=()) -> pd.DataFrame:
    conn = sqlite3.connect(":memory:")
    for name, df in tables.items():
        df.to_sql(name, conn, index=False)
    out = pd.read_sql_query(sql, conn, params=params)
    conn.close()
    return out


def rows(df: pd.DataFrame) -> list:
    return [[None if pd.isna(v) else v for v in r] for r in df.itertuples(index=False)]


def scores_df(values) -> pd.DataFrame:
    return pd.DataFrame({"id": range(1, len(values) + 1), "score": values})


EXAMPLE = scores_df([3.50, 3.65, 4.00, 3.85, 4.00, 3.65])
print(EXAMPLE)

   id  score
0   1   3.50
1   2   3.65
2   3   4.00
3   4   3.85
4   5   4.00
5   6   3.65


## Hàm cửa sổ: tính cho từng dòng mà không gộp dòng

`GROUP BY` **gộp** nhiều dòng thành một; hàm cửa sổ thì giữ nguyên số dòng và **thêm** một cột tính từ "cửa sổ" các dòng liên quan:

```sql
hàm() OVER (PARTITION BY cột_nhóm ORDER BY cột_sắp)
```

- `PARTITION BY` (tùy chọn): chia dòng thành nhóm; hàm tính **riêng** trong mỗi nhóm.
- `ORDER BY`: thứ tự trong nhóm để đánh số/xếp hạng.

In [16]:
print(run_sql("""
SELECT score,
       ROW_NUMBER() OVER (ORDER BY score DESC) AS row_num,
       RANK()       OVER (ORDER BY score DESC) AS rnk,
       DENSE_RANK() OVER (ORDER BY score DESC) AS dense
FROM Scores
ORDER BY score DESC
""", {"Scores": EXAMPLE}))

   score  row_num  rnk  dense
0   4.00        1    1      1
1   4.00        2    1      1
2   3.85        3    3      2
3   3.65        4    4      3
4   3.65        5    4      3
5   3.50        6    6      4


Đề yêu cầu đúng cột `dense`: `RANK()` nhảy từ 1 sang 3 sau hai điểm `4.00`, còn `DENSE_RANK()` thì không.

# 4. Cách 1: `DENSE_RANK()`

```sql
SELECT score,
       DENSE_RANK() OVER (ORDER BY score DESC) AS `rank`
FROM Scores
ORDER BY score DESC;
```

Trong MySQL 8, `RANK` là **từ khóa** (tên hàm) → tên cột phải bọc trong dấu huyền `` `rank` `` hoặc nháy `'rank'`. Trong SQLite dùng nháy kép `"rank"`.

In [17]:
SQL_DENSE = """
SELECT score,
       DENSE_RANK() OVER (ORDER BY score DESC) AS "rank"
FROM Scores
ORDER BY score DESC
"""
print(run_sql(SQL_DENSE, {"Scores": EXAMPLE}))

   score  rank
0   4.00     1
1   4.00     1
2   3.85     2
3   3.65     3
4   3.65     3
5   3.50     4


# 5. Cách 2: không dùng hàm cửa sổ — đếm số mức điểm khác nhau `>=` mình

Hạng dày của điểm `s` = số mức điểm **khác nhau** lớn hơn **hoặc bằng** `s`:

```text
điểm khác nhau giảm dần: 4.00, 3.85, 3.65, 3.50
hạng của 3.65 = |{4.00, 3.85, 3.65}| = 3
```

```sql
SELECT s1.score,
       (SELECT COUNT(DISTINCT s2.score) FROM Scores s2 WHERE s2.score >= s1.score) AS `rank`
FROM Scores s1
ORDER BY s1.score DESC;
```

Truy vấn con **tương quan** chạy một lần cho mỗi dòng → `O(n²)`, nhưng chạy được trên mọi phiên bản MySQL (kể cả 5.7 không có hàm cửa sổ).

In [18]:
SQL_CORR = """
SELECT s1.score,
       (SELECT COUNT(DISTINCT s2.score) FROM Scores s2 WHERE s2.score >= s1.score) AS "rank"
FROM Scores s1
ORDER BY s1.score DESC
"""
print(run_sql(SQL_CORR, {"Scores": EXAMPLE}))

   score  rank
0   4.00     1
1   4.00     1
2   3.85     2
3   3.65     3
4   3.65     3
5   3.50     4


# 6. Mô phỏng

In [19]:
def trace_rank(values):
    distinct_desc = sorted(set(values), reverse=True)
    print("các mức điểm khác nhau (giảm dần):", distinct_desc)
    for s in sorted(values, reverse=True):
        ge = [d for d in distinct_desc if d >= s]
        print(f"   điểm {s:.2f}: các mức >= nó = {ge} → hạng {len(ge)}")

trace_rank([3.50, 3.65, 4.00, 3.85, 4.00, 3.65])

các mức điểm khác nhau (giảm dần): [4.0, 3.85, 3.65, 3.5]
   điểm 4.00: các mức >= nó = [4.0] → hạng 1
   điểm 4.00: các mức >= nó = [4.0] → hạng 1
   điểm 3.85: các mức >= nó = [4.0, 3.85] → hạng 2
   điểm 3.65: các mức >= nó = [4.0, 3.85, 3.65] → hạng 3
   điểm 3.65: các mức >= nó = [4.0, 3.85, 3.65] → hạng 3
   điểm 3.50: các mức >= nó = [4.0, 3.85, 3.65, 3.5] → hạng 4


# 7. Phiên bản nộp LeetCode

**MySQL** — xem mục 4.

**Pandas:**

In [20]:
def order_scores(scores: pd.DataFrame) -> pd.DataFrame:
    out = scores[["score"]].copy()
    out["rank"] = scores["score"].rank(method="dense", ascending=False).astype(int)
    return out.sort_values("score", ascending=False).reset_index(drop=True)

print(order_scores(EXAMPLE))

   score  rank
0   4.00     1
1   4.00     1
2   3.85     2
3   3.65     3
4   3.65     3
5   3.50     4


In [21]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<16} PASS")


def expected_rank(values):
    d = sorted(set(values), reverse=True)
    pos = {s: i + 1 for i, s in enumerate(d)}
    return [[s, pos[s]] for s in sorted(values, reverse=True)]


exp = [[4.0, 1], [4.0, 1], [3.85, 2], [3.65, 3], [3.65, 3], [3.5, 4]]
check("Ví dụ (DENSE)", rows(run_sql(SQL_DENSE, {"Scores": EXAMPLE})), exp)
check("Ví dụ (đếm)", rows(run_sql(SQL_CORR, {"Scores": EXAMPLE})), exp)
check("Ví dụ (Pandas)", rows(order_scores(EXAMPLE)), exp)
check("Tên cột", list(run_sql(SQL_DENSE, {"Scores": EXAMPLE}).columns), ["score", "rank"])
check("Một dòng", rows(run_sql(SQL_DENSE, {"Scores": scores_df([2.5])})), [[2.5, 1]])

random.seed(178)
for _ in range(300):
    vals = [random.randint(0, 8) / 2 for _ in range(random.randint(1, 10))]
    e = expected_rank(vals)
    t = {"Scores": scores_df(vals)}
    assert rows(run_sql(SQL_DENSE, t)) == e == rows(run_sql(SQL_CORR, t)) == rows(order_scores(scores_df(vals))), vals
print("300 bảng ngẫu nhiên: hai câu SQL và bản Pandas khớp")

Ví dụ (DENSE)    PASS
Ví dụ (đếm)      PASS
Ví dụ (Pandas)   PASS
Tên cột          PASS
Một dòng         PASS


300 bảng ngẫu nhiên: hai câu SQL và bản Pandas khớp


Chú ý: các dòng **bằng điểm** có thể ra theo thứ tự bất kỳ; ta so `[score, rank]` nên không bị ảnh hưởng.

# 8. Vì sao đúng? Độ phức tạp?

`DENSE_RANK` theo định nghĩa = `1 +` số giá trị **khác nhau** đứng trước trong thứ tự sắp xếp = số giá trị khác nhau `>=` giá trị hiện tại. Đó đúng là quy tắc của đề (bằng nhau cùng hạng, hạng liền kề).

| Cách | Độ phức tạp |
|---|---|
| `DENSE_RANK()` | `O(n log n)` (sắp xếp) |
| Truy vấn con tương quan | `O(n²)` |

# 9. Những lỗi dễ mắc

## Lỗi 1: Dùng `RANK()` — hạng bị "nhảy"

In [22]:
SQL_RANK = SQL_DENSE.replace("DENSE_RANK()", "RANK()")
print("Sai :", [r[1] for r in rows(run_sql(SQL_RANK, {"Scores": EXAMPLE}))])
print("Đúng:", [r[1] for r in rows(run_sql(SQL_DENSE, {"Scores": EXAMPLE}))])

Sai : [1, 1, 3, 4, 4, 6]
Đúng: [1, 1, 2, 3, 3, 4]


## Lỗi 2: Đếm điểm **không** `DISTINCT` trong truy vấn con

In [23]:
SQL_CORR_NO_DISTINCT = SQL_CORR.replace("COUNT(DISTINCT s2.score)", "COUNT(s2.score)")
print("Sai :", [r[1] for r in rows(run_sql(SQL_CORR_NO_DISTINCT, {"Scores": EXAMPLE}))])
print("Đúng:", [r[1] for r in rows(run_sql(SQL_CORR, {"Scores": EXAMPLE}))])

Sai : [2, 2, 3, 5, 5, 6]
Đúng: [1, 1, 2, 3, 3, 4]


Đếm cả bản trùng thì hai điểm `4.00` cùng có hạng 2 — đó là "số người `>=` mình", không phải hạng dày.

## Lỗi 3: Dùng `>` thay vì `>=`

In [24]:
SQL_CORR_GT = SQL_CORR.replace("s2.score >= s1.score", "s2.score > s1.score")
print("Sai :", [r[1] for r in rows(run_sql(SQL_CORR_GT, {"Scores": EXAMPLE}))], "← lệch 1, điểm cao nhất có hạng 0")

Sai : [0, 0, 1, 2, 2, 3] ← lệch 1, điểm cao nhất có hạng 0


## Lỗi 4: Đặt tên cột `rank` không có dấu nháy trong MySQL 8

MySQL 8 báo lỗi cú pháp vì `RANK` là tên hàm dành riêng. Viết `` AS `rank` `` hoặc `AS 'rank'`.

## Lỗi 5: Quên `ORDER BY score DESC` ở cuối

`ORDER BY` bên trong `OVER (...)` chỉ quyết định cách **tính hạng**, không quyết định **thứ tự dòng kết quả**. Đề yêu cầu kết quả sắp theo điểm giảm dần → cần `ORDER BY` ở cuối truy vấn.

# 10. Mẹo nhận diện pattern

- **Xếp hạng có bằng nhau** → `DENSE_RANK` (không nhảy) hay `RANK` (nhảy) tùy đề; `ROW_NUMBER` khi cần số thứ tự duy nhất.
- **Xếp hạng trong từng nhóm** → thêm `PARTITION BY` (bài tập 4).
- **Hệ CSDL cũ không có hàm cửa sổ** → truy vấn con tương quan đếm.

Bài liên quan: **#176**, **#177**, **#184 Department Highest Salary**, **#185 Department Top Three Salaries** (Hard), **#1532**, **#2041**.

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Xếp hạng dày cho điểm `[2.0, 3.5, 3.5, 1.0, 2.0, 4.0]`. So với `RANK()`.

## Bài 2 — Viết câu SQL dùng `DENSE_RANK`

In [25]:
def exercise_2_sql() -> str:
    # TODO
    pass


# assert rows(run_sql(exercise_2_sql(), {"Scores": scores_df([3.50, 3.65, 4.00, 3.85, 4.00, 3.65])})) == [[4.0, 1], [4.0, 1], [3.85, 2], [3.65, 3], [3.65, 3], [3.5, 4]]

## Bài 3 — Viết câu SQL **không** dùng hàm cửa sổ

In [26]:
def exercise_3_sql() -> str:
    # TODO
    pass


# assert rows(run_sql(exercise_3_sql(), {"Scores": scores_df([1.0, 2.0, 2.0])})) == [[2.0, 1], [2.0, 1], [1.0, 2]]

## Bài 4 — Biến thể: xếp hạng trong từng lớp (`PARTITION BY`)

Bảng `Students(name, class, score)`. Trả về `class, name, score, rank` (hạng dày **trong lớp**), sắp theo `class` tăng dần, rồi `score` giảm dần, rồi `name` tăng dần.

In [27]:
STUDENTS = pd.DataFrame({
    "name":  ["An", "Bình", "Chi", "Dũng", "Em", "Giang"],
    "class": ["A",  "A",    "A",   "B",    "B",  "B"],
    "score": [9.0,  8.0,    9.0,   7.0,    8.5,  7.0],
})

def exercise_4_sql() -> str:
    # TODO
    pass


# assert rows(run_sql(exercise_4_sql(), {"Students": STUDENTS})) == [["A", "An", 9.0, 1], ["A", "Chi", 9.0, 1], ["A", "Bình", 8.0, 2], ["B", "Em", 8.5, 1], ["B", "Dũng", 7.0, 2], ["B", "Giang", 7.0, 2]]

## Bài 5 — Mini challenge: Pandas cho bài 4 (`groupby(...).rank`)

In [28]:
def exercise_5_rank_in_class(students: pd.DataFrame) -> pd.DataFrame:
    # TODO: trả về DataFrame cột class, name, score, rank theo thứ tự như bài 4
    pass


# assert rows(exercise_5_rank_in_class(STUDENTS)) == [["A", "An", 9.0, 1], ["A", "Chi", 9.0, 1], ["A", "Bình", 8.0, 2], ["B", "Em", 8.5, 1], ["B", "Dũng", 7.0, 2], ["B", "Giang", 7.0, 2]]

# 12. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_rank` và câu SQL ở mục 3 sau khi tự làm.

**Bài 4**
- Mức 1: `DENSE_RANK() OVER (PARTITION BY class ORDER BY score DESC)`.
- Mức 2: `ORDER BY class, score DESC, name`.

**Bài 5**
- Mức 1: `students.groupby("class")["score"].rank(method="dense", ascending=False)`.
- Mức 2: `sort_values(["class", "score", "name"], ascending=[True, False, True])`.

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [29]:
v1 = scores_df([2.0, 3.5, 3.5, 1.0, 2.0, 4.0])
print(run_sql("SELECT score, DENSE_RANK() OVER (ORDER BY score DESC) AS dense, RANK() OVER (ORDER BY score DESC) AS rnk FROM Scores ORDER BY score DESC", {"Scores": v1}))

   score  dense  rnk
0    4.0      1    1
1    3.5      2    2
2    3.5      2    2
3    2.0      3    4
4    2.0      3    4
5    1.0      4    6


## Đáp án Bài 2

In [30]:
def answer_2_sql() -> str:
    return SQL_DENSE

assert rows(run_sql(answer_2_sql(), {"Scores": scores_df([3.50, 3.65, 4.00, 3.85, 4.00, 3.65])})) == [[4.0, 1], [4.0, 1], [3.85, 2], [3.65, 3], [3.65, 3], [3.5, 4]]
print("Bài 2 PASS (giống SQL_DENSE ở mục 4)")

Bài 2 PASS (giống SQL_DENSE ở mục 4)


## Đáp án Bài 3

In [31]:
def answer_3_sql() -> str:
    return SQL_CORR

assert rows(run_sql(answer_3_sql(), {"Scores": scores_df([1.0, 2.0, 2.0])})) == [[2.0, 1], [2.0, 1], [1.0, 2]]
print("Bài 3 PASS (giống SQL_CORR ở mục 5)")

Bài 3 PASS (giống SQL_CORR ở mục 5)


## Đáp án Bài 4

In [32]:
def answer_4_sql() -> str:
    return """
    SELECT class, name, score,
           DENSE_RANK() OVER (PARTITION BY class ORDER BY score DESC) AS "rank"
    FROM Students
    ORDER BY class, score DESC, name
    """

print(run_sql(answer_4_sql(), {"Students": STUDENTS}))
assert rows(run_sql(answer_4_sql(), {"Students": STUDENTS})) == [["A", "An", 9.0, 1], ["A", "Chi", 9.0, 1], ["A", "Bình", 8.0, 2], ["B", "Em", 8.5, 1], ["B", "Dũng", 7.0, 2], ["B", "Giang", 7.0, 2]]
print("Bài 4 PASS")

  class   name  score  rank
0     A     An    9.0     1
1     A    Chi    9.0     1
2     A   Bình    8.0     2
3     B     Em    8.5     1
4     B   Dũng    7.0     2
5     B  Giang    7.0     2
Bài 4 PASS


## Đáp án Bài 5

In [33]:
def answer_5_rank_in_class(students: pd.DataFrame) -> pd.DataFrame:
    out = students[["class", "name", "score"]].copy()
    out["rank"] = students.groupby("class")["score"].rank(method="dense", ascending=False).astype(int)
    return out.sort_values(["class", "score", "name"], ascending=[True, False, True]).reset_index(drop=True)

assert rows(answer_5_rank_in_class(STUDENTS)) == [["A", "An", 9.0, 1], ["A", "Chi", 9.0, 1], ["A", "Bình", 8.0, 2], ["B", "Em", 8.5, 1], ["B", "Dũng", 7.0, 2], ["B", "Giang", 7.0, 2]]
print("Bài 5 PASS")

Bài 5 PASS


# 14. Tổng kết

> Hạng dày = `DENSE_RANK() OVER (ORDER BY score DESC)`; không có hàm cửa sổ thì đếm `COUNT(DISTINCT s2.score) WHERE s2.score >= s1.score`. Đặt tên cột `` `rank` `` trong dấu nháy (MySQL 8), và nhớ `ORDER BY score DESC` ở cuối truy vấn.

---

## Checklist trước khi nộp

- [ ] `DENSE_RANK` (không phải `RANK`)?
- [ ] Tên cột `rank` có dấu nháy?
- [ ] Kết quả sắp theo `score` giảm dần?
- [ ] Bản đếm: `COUNT(DISTINCT ...)` và `>=`?